In [3]:
import numpy as np
import k3d

In [4]:
class Detector:
    def __init__(self, R=2, N=40, r=0.1, a=0.5):
        self.R = R
        self.N = N
        self.r = r
        self.a = a

        self.pm_positions = self.generate_positions()

    def generate_positions(self):
        """Равномерное распределение фотоумножителей по сфере."""
        positions = []

        golden_angle = np.pi * (3 - np.sqrt(5))

        for i in range(self.N):
            z = 1 - 2 * (i + 0.5) / self.N
            theta = golden_angle * i

            rho = np.sqrt(1 - z**2)

            x = self.R * rho * np.cos(theta)
            y = self.R * rho * np.sin(theta)
            z = self.R * z

            positions.append([x, y, z])

        return np.array(positions)

    def is_on_detector(self, point, tol=1e-6):
        """Проверяет, лежит ли точка на поверхности детектора."""
        point = np.array(point)
        return abs(np.linalg.norm(point) - self.R) < tol

    def belongs_to_pm(self, point):
        """Проверяет, принадлежит ли точка фотоумножителю."""
        point = np.array(point)

        if not self.is_on_detector(point):
            return False

        cos_angles = self.pm_positions @ point / self.R**2
        cos_angles = np.clip(cos_angles, -1, 1)

        distances = self.R * np.arccos(cos_angles)

        return np.any(distances <= self.r)

    def visualize(self):
        """Визуализирует детектор и фотоумножители."""

        # Создаём сетку для сферической поверхности
        n_theta = 50
        n_phi = 50

        theta = np.linspace(0, np.pi, n_theta)
        phi = np.linspace(0, 2 * np.pi, n_phi)

        theta, phi = np.meshgrid(theta, phi)

        x = self.R * np.sin(theta) * np.cos(phi)
        y = self.R * np.sin(theta) * np.sin(phi)
        z = self.R * np.cos(theta)

        vertices = np.column_stack((
            x.flatten(),
            y.flatten(),
            z.flatten()
        )).astype(np.float32)

        indices = []

        for i in range(n_theta - 1):
            for j in range(n_phi - 1):
                p1 = i * n_phi + j
                p2 = p1 + n_phi
                p3 = p1 + 1
                p4 = p2 + 1

                indices.append([p1, p2, p3])
                indices.append([p2, p4, p3])

        indices = np.array(indices, dtype=np.uint32)

        plot = k3d.plot()

        # Поверхность детектора
        plot += k3d.mesh(
            vertices,
            indices,
            color=0x4488FF,
            opacity=0.15
        )

        # Фотоумножители
        plot += k3d.points(
            positions=self.pm_positions.astype(np.float32),
            point_size=2 * self.r,
            color=0xFF0000,
            shader='3d'
        )

        plot.display()

In [5]:
# Создание и визуализация детектора
detector = Detector()
detector.visualize()

Output()

In [6]:
# Проверяем центр первого фотоумножителя
point = detector.pm_positions[0]

print(detector.belongs_to_pm(point))  # True

# Проверяем точку на поверхности вдали от фотоумножителей
point = np.array([0, 0, detector.R])

print(detector.belongs_to_pm(point))

True
False
